## Learning Advanced RAG

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
import pandas as pd
import math

In [2]:
# FORMULAS
FORMULAS = {
    "SI1": "SI1 (%) = ((1.5 - 1.6) / 1.5) × 100%",
    "SI4": "SI4 = (1.5 - 1.6) / (1.11 + 1.13)",
    "EC2": "EC2 (%) = (2.2 / 1.7) × 100",
    "EC4": "EC4 = 2.6 / (1.11 + 1.13)",
    "EC8": "EC8 = 2.11 / (1.11 + 1.13)",
    "TR1": "TR1 = (5.1 + 5.2 + 5.3) / (1.11 + 1.13)",
    "TR4": "TR4 = 5.10 / (1.11 + 1.13)",
    "TR5": "TR5 (%) = (5.12 / 1.5) × 100%",
    "ED1": "ED1 (%) = (6.1 / 6.2) × 100%",
    "ED2": "ED2 (%) = (6.5 / 6.6) × 100%",
    "ED3": "ED3 = 6.9 / 6.8",
    "ED10": "ED10 (%) = (6.17 / 6.18) × 100%",
    "GD1": "GD1 (%) = (7.2 / 7.1) × 100%",
    "GD8": "GD8 (%) = (7.16 / 7.15) × 100%",
}

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
api_key_env = os.getenv("DEEPSEEK_API_KEY")
if not api_key_env:
    print("ERROR: API Key tidak ditemukan! Pastikan file .env sudah benar.")
else:
    print("API Key berhasil dimuat.")

client = OpenAI(
    api_key=api_key_env, 
    base_url="https://api.deepseek.com"
)

API Key berhasil dimuat.


In [5]:
def generate_real_answer(pertanyaan_user, konteks_pdf):
    prompt_system = """You are an UI GreenMetric AI assistant.
    You will be given a question and a context from the UI GreenMetric Guideline.
    Your answers should be based solely on the provided context, and you should not use any external information.
    If the answer to the question is not found in the context, respond with "Sorry, I don't know." and do not provide any additional information.
    Always provide a concise and accurate answer based on the context, and avoid including any information that is not explicitly stated in the context.
    """
    
    prompt_user = f"KONTEKS:\n{konteks_pdf}\n\nPERTANYAAN: {pertanyaan_user}"

    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": prompt_system},
            {"role": "user", "content": prompt_user}
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content

#### Embed Model

In [6]:
print("Embedded model used: (all-MiniLM-L6-v2)")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Embedded model used: (all-MiniLM-L6-v2)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    
    print(f"PDF to extract: {pdf_path}")
    print(f"Pages: {len(reader.pages)}")
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            full_text += text + "\n"
            
    print("Successful!")
    return full_text

path_file = "guidelines.pdf"
raw_text_full = extract_text_from_pdf(path_file)

print(f"\nTotal Characters: {len(raw_text_full)}")
print("-" * 30)
print(raw_text_full[:1000])

PDF to extract: guidelines.pdf
Pages: 1
Successful!

Total Characters: 1487
------------------------------
1. What is UI GreenMetric Sustainable University Rankings?    
Universitas Indonesia (UI) initiated a world university ranking in 2010, known 
through 2025 as the UI GreenMetric World University Rankings, to measure 
universities’ sustainability efforts. It was designed as an online survey to capture 
sustainability policies and programs implemented by universities across the globe.   
The ranking is broadly based on the conceptual framework of Environment, 
Economy, and Equity. Its categories, indicators, and weightings are intended to be 
relevant to a wide range of stakeholders while minimizing bias as much as 
possible. Data collection and submission are designed to be relatively 
straightforward and typically require a reasonable amount of staff time.   
In the 2010 edition, 95 universities from 35 countries participated: 18 from America, 
35 from Europe, 40 from Asia, and tw

In [8]:
def simple_recursive_chunker(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = start + chunk_size
        
        if end < text_len:
            last_dot = text.rfind('.', start, end)
            if last_dot != -1 and last_dot > start:
                end = last_dot + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap if end < text_len else text_len        
    return chunks

In [9]:
df = pd.read_csv('appendix1_questionnairemasterandscoring.csv', sep=';', encoding='latin-1', dtype={'no': str})

def is_null(val):
    if val is None:
        return True
    try:
        return math.isnan(float(val))
    except (ValueError, TypeError):
        return str(val).strip() in ('N/A', 'nan', '')

csv_chunks = []
for no, group in df.groupby('no', sort=False):
    first = group.iloc[0]
    options = []
    for _, row in group.iterrows():
        if not is_null(row['max_score']):
            options.append(f"  {row['answer']} (max_score: {row['max_score']}, calculated_score: {row['calculated_score']})")
        else:
            options.append(f"  {row['answer']}")
    
    indicator_code = None if is_null(first['indicator_code']) else first['indicator_code']
    formula_line = f"\nFormula: {FORMULAS.get(indicator_code)}" if indicator_code and indicator_code in FORMULAS else ""
    indicator_line = indicator_code if indicator_code else "None"

    chunk_text = f"""Question {no} — {first['criteria']}
Category: {first['category']}
Indicator Code: {indicator_line}{formula_line}
Evidence Required: {first['evidence_required']}
Options:
{chr(10).join(options)}"""

    csv_chunks.append({
        "teks_konten": chunk_text,
        "metadata": {
            "sumber": "appendix1_questionnairemasterandscoring.csv",
            "kategori": first['category'],
            "question_no": no,
            "indicator_code": indicator_code if indicator_code else "None",
            "evidence_required": first['evidence_required']
        }
    })

print(f"CSV chunks created: {len(csv_chunks)}")

CSV chunks created: 118


In [10]:
chunks_dari_pdf = simple_recursive_chunker(raw_text_full, chunk_size=500, overlap=100)

documents = []
for i, chunk in enumerate(chunks_dari_pdf):
    documents.append({
        "teks_konten": chunk,
        "metadata": {
            "sumber": path_file,
            "chunk_id": i + 1,
            "kategori": "General"
        }
    })

print(f"Berhasil membuat {len(documents)} chunks siap pakai!")

Berhasil membuat 4 chunks siap pakai!


In [11]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="greenmetric_guidelines", metadata={"hnsw:space": "cosine"})

print("Menyimpan chunks PDF ke ChromaDB...")
for i, doc in enumerate(documents):
    doc["vektor"] = embed_model.encode(doc["teks_konten"])
    collection.add(
        documents=[doc["teks_konten"]],
        metadatas=[doc["metadata"]],
        ids=[f"id_{i}"],
        embeddings=[doc["vektor"].tolist()]
    )

print(f"Selesai! {len(documents)} chunks PDF tersimpan.")

print("Menyimpan chunks CSV ke ChromaDB...")
for i, doc in enumerate(csv_chunks):
    doc["vektor"] = embed_model.encode(doc["teks_konten"])
    collection.add(
        documents=[doc["teks_konten"]],
        metadatas=[doc["metadata"]],
        ids=[f"csv_{i}"],
        embeddings=[doc["vektor"].tolist()] #
    )

print(f"Selesai! {len(csv_chunks)} CSV chunks tersimpan.")

Menyimpan chunks PDF ke ChromaDB...
Selesai! 4 chunks PDF tersimpan.
Menyimpan chunks CSV ke ChromaDB...
Selesai! 118 CSV chunks tersimpan.


In [12]:
def find_document(pertanyaan, top_k=1, threshold=None):
    query_vector = embed_model.encode(pertanyaan).tolist()
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )

    if not results['documents'][0]:
        return []

    valid_results = []
    
    for i in range(len(results['documents'][0])):
        distance = results['distances'][0][i]

        if threshold is not None and distance > threshold+(i*0.012):  # Threshold tolerance
            break
            
        valid_results.append({
            "teks": results['documents'][0][i],
            "skor": distance,
            "sumber": results['metadatas'][0][i].get('sumber', 'Unknown')
        })
    
    return valid_results

#### Naive search (no threshold)

In [13]:
list_questions = ["What's the theme for UI GreenMetric in 2026?", "I want to go to the toilet to take a piss"]
for questions in list_questions:
    print(f"\nPertanyaan: {questions}")
    hasil = find_document(questions)
    if hasil:
        print(f"Hasil ditemukan!")
        print(f"Jarak (Distance): {hasil[0]['skor']:.4f} (Semakin kecil semakin baik)")
        print(f"Isi Dokumen: {hasil[0]['teks']}")
        print(f"Sumber: {hasil[0]['sumber']}")
    else:
        print("Tidak ditemukan dokumen yang relevan.")


Pertanyaan: What's the theme for UI GreenMetric in 2026?
Hasil ditemukan!
Jarak (Distance): 0.4591 (Semakin kecil semakin baik)
Isi Dokumen: lobal 
recognition of the UI GreenMetric as a pioneering sustainability-focused university 
ranking.   
Starting in 2026, the ranking was renamed the UI GreenMetric Sustainable 
University Rankings. The theme for 2026 is “Advancing Sustainable Campuses 
through Governance, Digitalization, and Integrated Performance.” This highlights 
how universities strengthen sustainability through effective governance structures, 
transparent leadership, and accountable policies.
Sumber: guidelines.pdf

Pertanyaan: I want to go to the toilet to take a piss
Hasil ditemukan!
Jarak (Distance): 0.7973 (Semakin kecil semakin baik)
Isi Dokumen: Question 3.17 — Sewage disposal
Category: Waste (WS)
Indicator Code: WS6
Evidence Required: Yes
Options:
  [1] Untreated into waterways (max_score: 300.0, calculated_score: 0x300)
  [2] Treated with preliminary treatment (max

#### Adding threshold and testings

In [14]:
test_threshold = 0.7
test_top_k = 5
def tanya_bot_greenmetric(tanya):
    print(f"User: {tanya}")
    
    doc_match = find_document(tanya, top_k=test_top_k, threshold=test_threshold)
    
    if not doc_match:
        doc_match = find_document(tanya, top_k=test_top_k)
        print(f"Log: Tidak ada dokumen yang ditemukan (score: {doc_match[0]['skor']:.4f})")
        return "INFORMATION NOT FOUND IN DOCUMENT/CONTEXT."
    
    print(f"Log: Jumlah dokummen ditemukan: {len(doc_match)} (score: {doc_match[0]['skor']:.4f})")
    #jawaban = generate_real_answer(tanya, doc_match[0]['teks'])

    contexts = [doc_match[i]['teks'] for i in range(0, len(doc_match))]
    jawaban = generate_real_answer(tanya, contexts)
    
    return f"\nBOT: {jawaban}\n(Sumber: {doc_match[0]['sumber']})"

In [15]:
print(tanya_bot_greenmetric("What's UI GreenMetric's 2026 theme?"))

User: What's UI GreenMetric's 2026 theme?
Log: Jumlah dokummen ditemukan: 5 (score: 0.5010)

BOT: The theme for 2026 is “Advancing Sustainable Campuses through Governance, Digitalization, and Integrated Performance.”
(Sumber: guidelines.pdf)


In [16]:
print(tanya_bot_greenmetric("How do I submit my university's data for the UI GreenMetric ranking?"))

User: How do I submit my university's data for the UI GreenMetric ranking?
Log: Jumlah dokummen ditemukan: 5 (score: 0.3351)

BOT: Sorry, I don't know.
(Sumber: guidelines.pdf)


In [17]:
print(tanya_bot_greenmetric("Oh no my laptop is broken, please help me!"))

User: Oh no my laptop is broken, please help me!
Log: Tidak ada dokumen yang ditemukan (score: 0.9107)
INFORMATION NOT FOUND IN DOCUMENT/CONTEXT.


In [18]:
print(tanya_bot_greenmetric("What is the formula for calculating SI1?"))

User: What is the formula for calculating SI1?
Log: Jumlah dokummen ditemukan: 5 (score: 0.6653)

BOT: The formula for SI1 is: SI1 (%) = ((1.5 - 1.6) / 1.5) × 100%.
(Sumber: appendix1_questionnairemasterandscoring.csv)


In [19]:
print(tanya_bot_greenmetric("What are the questions with most options?"))

User: What are the questions with most options?
Log: Tidak ada dokumen yang ditemukan (score: 0.7421)
INFORMATION NOT FOUND IN DOCUMENT/CONTEXT.


In [20]:
print(tanya_bot_greenmetric("Settings and Infrastructure category, how many questions are there?"))

User: Settings and Infrastructure category, how many questions are there?
Log: Jumlah dokummen ditemukan: 5 (score: 0.4522)

BOT: Based on the provided context, there are 5 questions listed under the Setting and Infrastructure (SI) category.
(Sumber: appendix1_questionnairemasterandscoring.csv)
